<a href="https://colab.research.google.com/github/22417010/Gemini-Intelligence-Benchmarking/blob/main/11rag_long_context_demo1_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

تثبيت المكتبات


النتيجة المتوقعة:
تثبيت أدوات الواجهة، FAISS، embeddings، وقراءة PDF.

In [2]:
!pip install -q gradio faiss-cpu sentence-transformers numpy python-dotenv requests pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.9 MB/s eta 0:00:00


إنشاء هيكل المشروع

النتيجة المتوقعة:

rag_long_context_demo/
├── src/
└── data/

In [3]:
import os

PROJECT_NAME = "rag_long_context_demo"

folders = [
    PROJECT_NAME,
    f"{PROJECT_NAME}/src",
    f"{PROJECT_NAME}/data"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


إنشاء ملف requirements.txt



In [4]:
requirements = """
gradio
faiss-cpu
sentence-transformers
numpy
python-dotenv
requests
pypdf
"""

with open(f"{PROJECT_NAME}/requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt created.")

requirements.txt created.


إنشاء ملف الإعدادات config.py

وظيفته:
هذا الملف يجمع الإعدادات بدل ما نكررها في كل مكان.

In [5]:
config_code = """
import os

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

OPENROUTER_MODEL = os.getenv(
    "OPENROUTER_MODEL",
    "openai/gpt-4o-mini"
)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 3
"""

with open(f"{PROJECT_NAME}/src/config.py", "w") as f:
    f.write(config_code)

print("config.py created.")

config.py created.


إنشاء عميل OpenRouter

ملف llm_client.py

وظيفته:


هذا الملف مسؤول فقط عن إرسال prompt إلى OpenRouter وإرجاع الإجابة.

In [6]:
llm_client_code = """
import requests
from .config import OPENROUTER_API_KEY, OPENROUTER_MODEL

def call_llm(prompt, temperature=0.2, max_tokens=700):
    if not OPENROUTER_API_KEY:
        return "ERROR: OPENROUTER_API_KEY is missing. Please set it before running the app."

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful academic assistant. Answer clearly and accurately."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)

        if response.status_code != 200:
            return f"API Error {response.status_code}: {response.text}"

        data = response.json()
        return data["choices"][0]["message"]["content"]

    except Exception as e:
        return f"Request failed: {str(e)}"
"""

with open(f"{PROJECT_NAME}/src/llm_client.py", "w") as f:
    f.write(llm_client_code)

print("llm_client.py created.")

llm_client.py created.


إعداد مفتاح OpenRouter داخل Colab

In [7]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-4o-mini"

print("API key configured successfully.")

Enter your OpenRouter API Key: ··········
API key configured successfully.


المرحلة 4: قراءة الملفات والوثائق


إنشاء documents.py

In [8]:
documents_code = """
from pypdf import PdfReader


SAMPLE_DOCUMENTS = [
    {
        "title": "RAG",
        "text": '''
Retrieval-Augmented Generation is a technique that combines information retrieval
with language generation. Instead of sending all documents to the model, RAG
retrieves only the most relevant chunks and uses them as context.
'''
    },
    {
        "title": "Long Context",
        "text": '''
Long Context models process a large amount of text directly inside the prompt.
This can be useful when the model needs to see the whole document, but it may
increase cost and latency.
'''
    },
    {
        "title": "FAISS",
        "text": '''
FAISS is a library developed for efficient similarity search over dense vectors.
It is commonly used in RAG systems to retrieve semantically similar chunks.
'''
    }
]


def read_txt_file(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def read_pdf_file(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)

    return "\\n".join(pages)


def load_uploaded_files(files):
    documents = []

    if not files:
        return documents

    for file in files:
        file_path = file.name
        file_name = file_path.split("/")[-1]

        if file_path.lower().endswith(".pdf"):
            text = read_pdf_file(file_path)
        elif file_path.lower().endswith(".txt"):
            text = read_txt_file(file_path)
        else:
            text = ""

        if text.strip():
            documents.append({
                "title": file_name,
                "text": text
            })

    return documents


def get_documents(uploaded_files=None, use_sample_docs=True):
    documents = []

    if use_sample_docs:
        documents.extend(SAMPLE_DOCUMENTS)

    uploaded_docs = load_uploaded_files(uploaded_files)
    documents.extend(uploaded_docs)

    return documents
"""

with open(f"{PROJECT_NAME}/src/documents.py", "w") as f:
    f.write(documents_code)

print("documents.py created.")

documents.py created.


**تقسيم النصوص إلى Chunks**

وظيفته:
يحوّل كل وثيقة طويلة إلى أجزاء صغيرة مناسبة للـRAG.

In [9]:
embeddings_code = """
from sentence_transformers import SentenceTransformer
from .config import EMBEDDING_MODEL


_model = None


def get_embedding_model():
    global _model

    if _model is None:
        _model = SentenceTransformer(EMBEDDING_MODEL)

    return _model


def encode_texts(texts):
    model = get_embedding_model()

    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings
"""

with open(f"{PROJECT_NAME}/src/embeddings.py", "w") as f:
    f.write(embeddings_code)

print("embeddings.py created.")

embeddings.py created.


In [10]:
import os

PROJECT_NAME = "rag_long_context_demo"

print(os.listdir(PROJECT_NAME))
print(os.listdir(f"{PROJECT_NAME}/src"))

['data', 'requirements.txt', 'src']
['config.py', 'llm_client.py', 'documents.py', 'embeddings.py']


In [11]:
PROJECT_NAME = "rag_long_context_demo"

chunking_code = """
from .config import CHUNK_SIZE, CHUNK_OVERLAP


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = text.replace("\\n", " ").strip()

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


def build_chunks(documents):
    all_chunks = []

    for doc_id, doc in enumerate(documents):
        chunks = chunk_text(doc["text"])

        for chunk_id, chunk in enumerate(chunks):
            all_chunks.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "title": doc["title"],
                "text": chunk
            })

    return all_chunks
"""

with open(f"{PROJECT_NAME}/src/chunking.py", "w", encoding="utf-8") as f:
    f.write(chunking_code)

print("chunking.py created successfully.")

chunking.py created successfully.


**المرحلة 6: إنشاء Embeddings**

استخدمنا model متعدد اللغات مناسب أكثر للعربية من

all-MiniLM-L6-v2.

In [12]:
embeddings_code = """
from sentence_transformers import SentenceTransformer
from .config import EMBEDDING_MODEL


_model = None


def get_embedding_model():
    global _model

    if _model is None:
        _model = SentenceTransformer(EMBEDDING_MODEL)

    return _model


def encode_texts(texts):
    model = get_embedding_model()

    embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings
"""

with open(f"{PROJECT_NAME}/src/embeddings.py", "w") as f:
    f.write(embeddings_code)

print("embeddings.py created.")

embeddings.py created.


**المرحلة 7: بناء FAISS Vector Store**


In [13]:
vector_store_code = """
import faiss
import numpy as np
from .embeddings import encode_texts
from .config import TOP_K


class FaissVectorStore:
    def __init__(self):
        self.index = None
        self.chunks = []
        self.dimension = None

    def build(self, chunks):
        self.chunks = chunks

        texts = [chunk["text"] for chunk in chunks]
        embeddings = encode_texts(texts).astype("float32")

        self.dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings)

    def search(self, query, top_k=TOP_K):
        if self.index is None or len(self.chunks) == 0:
            return []

        query_embedding = encode_texts([query]).astype("float32")

        scores, indices = self.index.search(query_embedding, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            chunk = self.chunks[idx].copy()
            chunk["score"] = float(score)
            results.append(chunk)

        return results
"""

with open(f"{PROJECT_NAME}/src/vector_store.py", "w") as f:
    f.write(vector_store_code)

print("vector_store.py created.")

vector_store.py created.


**المرحلة 8: اختبار سريع قبل التطبيق**

النتيجة المتوقعة:

يجب أن يرجع chunk متعلق بـFAISS.

**New Cell Added: Creating `src/chunking.py`**

In [14]:
import sys
sys.path.append(PROJECT_NAME)

from src.documents import get_documents
from src.chunking import build_chunks
from src.vector_store import FaissVectorStore

docs = get_documents(uploaded_files=None, use_sample_docs=True)
chunks = build_chunks(docs)

store = FaissVectorStore()
store.build(chunks)

results = store.search("What is FAISS used for?")

for r in results:
    print("Title:", r["title"])
    print("Score:", r["score"])
    print("Text:", r["text"])
    print("-" * 50)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Title: FAISS
Score: 0.43763676285743713
Text: FAISS is a library developed for efficient similarity search over dense vectors. It is commonly used in RAG systems to retrieve semantically similar chunks.
--------------------------------------------------
Title: RAG
Score: 0.17847689986228943
Text: Retrieval-Augmented Generation is a technique that combines information retrieval with language generation. Instead of sending all documents to the model, RAG retrieves only the most relevant chunks and uses them as context.
--------------------------------------------------
Title: Long Context
Score: 0.13598088920116425
Text: Long Context models process a large amount of text directly inside the prompt. This can be useful when the model needs to see the whole document, but it may increase cost and latency.
--------------------------------------------------


**الخلية 12: إنشاء rag_pipeline.py**

In [15]:
PROJECT_NAME = "rag_long_context_demo"

rag_pipeline_code = """
from .llm_client import call_llm
from .config import TOP_K


def build_rag_prompt(question, retrieved_chunks):
    context = "\\n\\n".join([
        f"Source: {chunk['title']}\\nContent: {chunk['text']}"
        for chunk in retrieved_chunks
    ])

    prompt = f'''
You are answering using Retrieval-Augmented Generation (RAG).

Question:
{question}

Retrieved Context:
{context}

Instructions:
- Answer only based on the retrieved context.
- If the context is insufficient, say that clearly.
- Provide a clear academic answer.
'''

    return prompt


def answer_with_rag(question, vector_store, top_k=TOP_K):
    retrieved_chunks = vector_store.search(question, top_k=top_k)

    if not retrieved_chunks:
        return {
            "answer": "No relevant chunks were retrieved.",
            "retrieved_chunks": []
        }

    prompt = build_rag_prompt(question, retrieved_chunks)
    answer = call_llm(prompt)

    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }
"""

with open(f"{PROJECT_NAME}/src/rag_pipeline.py", "w", encoding="utf-8") as f:
    f.write(rag_pipeline_code)

print("rag_pipeline.py created.")

rag_pipeline.py created.


**الخلية 13: إنشاء long_context.py**

In [16]:
long_context_code = """
from .llm_client import call_llm


def build_long_context_prompt(question, documents):
    full_context = "\\n\\n".join([
        f"Document: {doc['title']}\\nContent: {doc['text']}"
        for doc in documents
    ])

    prompt = f'''
You are answering using the Long Context approach.

Question:
{question}

Full Documents Context:
{full_context}

Instructions:
- Use the full provided context.
- Provide a clear academic answer.
- Mention if the answer depends on information spread across documents.
'''

    return prompt


def answer_with_long_context(question, documents):
    if not documents:
        return "No documents were provided."

    prompt = build_long_context_prompt(question, documents)
    answer = call_llm(prompt)

    return answer
"""

with open(f"{PROJECT_NAME}/src/long_context.py", "w", encoding="utf-8") as f:
    f.write(long_context_code)

print("long_context.py created.")

long_context.py created.


**الخلية 14: إنشاء comparison.py**

In [17]:
comparison_code = """
def compare_methods(question, rag_result, long_context_answer, documents, chunks):
    retrieved_count = len(rag_result.get("retrieved_chunks", []))
    total_docs = len(documents)
    total_chunks = len(chunks)

    comparison = f'''
## Final Comparison: RAG vs Long Context

### User Question
{question}

### RAG Method
- Uses only the most relevant retrieved chunks.
- Retrieved chunks used: {retrieved_count}
- Total available chunks: {total_chunks}
- Strength: More efficient for large document collections.
- Limitation: May miss information if retrieval fails.

### Long Context Method
- Sends the complete document context to the model.
- Total documents used: {total_docs}
- Strength: Gives the model access to all provided content.
- Limitation: More expensive and slower for large documents.

### Practical Conclusion
- Use RAG when documents are large, many, or frequently updated.
- Use Long Context when the full document is small enough and global understanding is needed.
- In real-world academic systems, RAG is usually more scalable.
'''

    return comparison
"""

with open(f"{PROJECT_NAME}/src/comparison.py", "w", encoding="utf-8") as f:
    f.write(comparison_code)

print("comparison.py created.")

comparison.py created.


الخلية 15: اختبار RAG وLong Context بدون واجهة

In [18]:
import sys, os

PROJECT_NAME = "rag_long_context_demo"
PROJECT_PATH = os.path.abspath(PROJECT_NAME)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

from src.documents import get_documents
from src.chunking import build_chunks
from src.vector_store import FaissVectorStore
from src.rag_pipeline import answer_with_rag
from src.long_context import answer_with_long_context
from src.comparison import compare_methods

question = "What is the difference between RAG and Long Context?"

documents = get_documents(uploaded_files=None, use_sample_docs=True)
chunks = build_chunks(documents)

store = FaissVectorStore()
store.build(chunks)

rag_result = answer_with_rag(question, store)
long_answer = answer_with_long_context(question, documents)
comparison = compare_methods(question, rag_result, long_answer, documents, chunks)

print("===== RAG ANSWER =====")
print(rag_result["answer"])

print("\n===== LONG CONTEXT ANSWER =====")
print(long_answer)

print("\n===== COMPARISON =====")
print(comparison)

===== RAG ANSWER =====
Retrieval-Augmented Generation (RAG) and Long Context models differ primarily in their approach to handling information for language generation tasks. 

RAG combines information retrieval with language generation by retrieving only the most relevant chunks of information from a larger set of documents. This means that instead of processing all available documents, RAG focuses on a select few that are deemed most pertinent to the task at hand, which can enhance efficiency and relevance in the generated output.

In contrast, Long Context models are designed to process a large amount of text directly within the prompt. This approach allows the model to consider the entirety of a document, which can be beneficial for tasks that require comprehensive understanding. However, this method may lead to increased costs and latency due to the larger volume of data being processed at once.

In summary, RAG emphasizes selective retrieval of relevant information, while Long Con

In [19]:
comparison_code = """
def compare_methods(question, rag_result, long_context_answer, documents, chunks):
    retrieved_chunks = rag_result.get("retrieved_chunks", [])

    retrieved_count = len(retrieved_chunks)
    total_docs = len(documents)
    total_chunks = len(chunks)

    long_context_length = sum(len(doc["text"].split()) for doc in documents)

    retrieved_preview = "\\n\\n".join([
        f"- {c['title']} (score={round(c['score'], 3)})"
        for c in retrieved_chunks
    ])

    comparison = f'''
## Final Comparison: RAG vs Long Context

### User Question
{question}

---

### RAG Method
- Retrieved chunks used: {retrieved_count}
- Total available chunks: {total_chunks}

Retrieved Sources:
{retrieved_preview}

✔ Efficient (uses only relevant data)
✔ Faster in large-scale systems
❌ Depends on retrieval quality

---

### Long Context Method
- Total documents used: {total_docs}
- Approx context size: {long_context_length} words

✔ Full context visibility
✔ Better for small datasets
❌ Expensive and slower with large inputs

---

### Key Insight (Very Important)

RAG = Smart Filtering
Long Context = Brute Force

---

### Final Recommendation

- Use RAG → real-world systems (search engines, chatbots, research tools)
- Use Long Context → small documents or deep analysis tasks
'''

    return comparison
"""

الخلية 17: إنشاء app.py

In [20]:
PROJECT_NAME = "rag_long_context_demo"

app_code = """
import os
import sys
import gradio as gr

sys.path.insert(0, os.path.abspath("."))

from src.documents import get_documents
from src.chunking import build_chunks
from src.vector_store import FaissVectorStore
from src.rag_pipeline import answer_with_rag
from src.long_context import answer_with_long_context
from src.comparison import compare_methods


def format_retrieved_chunks(chunks):
    if not chunks:
        return "No chunks retrieved."

    output = ""

    for i, chunk in enumerate(chunks, start=1):
        output += f'''
### Chunk {i}
**Source:** {chunk["title"]}
**Similarity Score:** {round(chunk["score"], 4)}

{chunk["text"]}

---
'''

    return output


def run_demo(question, uploaded_files, use_sample_docs, top_k):
    if not question or not question.strip():
        return "Please enter a question.", "", "", ""

    documents = get_documents(
        uploaded_files=uploaded_files,
        use_sample_docs=use_sample_docs
    )

    if not documents:
        return "No documents provided.", "", "", ""

    chunks = build_chunks(documents)

    if not chunks:
        return "No text chunks could be created.", "", "", ""

    vector_store = FaissVectorStore()
    vector_store.build(chunks)

    rag_result = answer_with_rag(
        question=question,
        vector_store=vector_store,
        top_k=int(top_k)
    )

    long_context_answer = answer_with_long_context(
        question=question,
        documents=documents
    )

    comparison = compare_methods(
        question=question,
        rag_result=rag_result,
        long_context_answer=long_context_answer,
        documents=documents,
        chunks=chunks
    )

    retrieved_chunks_md = format_retrieved_chunks(
        rag_result.get("retrieved_chunks", [])
    )

    return (
        rag_result["answer"],
        long_context_answer,
        comparison,
        retrieved_chunks_md
    )


with gr.Blocks(title="RAG vs Long Context Demo") as demo:
    gr.Markdown(
        '''
# RAG vs Long Context Demo

This academic demo compares two approaches:

- **RAG with FAISS**
- **Long Context Prompting**

Upload PDF/TXT files, ask a question, and compare the outputs.
'''
    )

    with gr.Row():
        with gr.Column(scale=1):
            question = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is the difference between RAG and Long Context?",
                lines=3
            )

            uploaded_files = gr.File(
                label="Upload PDF/TXT files",
                file_count="multiple",
                file_types=[".pdf", ".txt"]
            )

            use_sample_docs = gr.Checkbox(
                label="Use sample documents",
                value=True
            )

            top_k = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Top-K Retrieved Chunks"
            )

            run_button = gr.Button("Run Comparison", variant="primary")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("RAG Answer"):
                    rag_output = gr.Markdown()

                with gr.Tab("Long Context Answer"):
                    long_output = gr.Markdown()

                with gr.Tab("Final Comparison"):
                    comparison_output = gr.Markdown()

                with gr.Tab("Retrieved Chunks"):
                    chunks_output = gr.Markdown()

    run_button.click(
        fn=run_demo,
        inputs=[
            question,
            uploaded_files,
            use_sample_docs,
            top_k
        ],
        outputs=[
            rag_output,
            long_output,
            comparison_output,
            chunks_output
        ]
    )


if __name__ == "__main__":
    demo.launch(
        share=True,
        debug=True
    )
"""

with open(f"{PROJECT_NAME}/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created.")

app.py created.


الخلية 18: تشغيل التطبيق

In [21]:
%cd rag_long_context_demo
!python app.py

/content/rag_long_context_demo
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://79b390b8f9dbf63eb7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Loading weights: 100% 199/199 [00:00<00:00, 958.83it/s, Materializing param=pooler.dense.weight] 
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
exit
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://79b390b8f9dbf63eb7.gradio.live


This demo compares RAG and Long Context using the same question and the same documents.

RAG first retrieves the most relevant chunks using FAISS, then sends only those chunks to the LLM.

Long Context sends the complete document content directly to the LLM.

The final tab shows the practical engineering difference between both methods.

 **🥇 المرحلة 1: تجهيز المشروع للرفع
الخلية 19: ضغط المشروع**

In [23]:
import os
print(os.getcwd())
print(os.listdir())

/content/rag_long_context_demo
['data', 'app.py', 'requirements.txt', '.gradio', 'src']


المرحلة 2: رفعه على GitHub
الخطوات:
1. ادخل GitHub

👉 https://github.com

In [24]:
import shutil
import os

PROJECT_PATH = "/content/rag_long_context_demo"

if os.path.exists(PROJECT_PATH):
    shutil.make_archive("rag_demo_project", 'zip', PROJECT_PATH)
    print("✅ Project zipped successfully.")
else:
    print("❌ Folder not found. Check path:", PROJECT_PATH)

✅ Project zipped successfully.


In [25]:
%cd /content/rag_long_context_demo
!ls

/content/rag_long_context_demo
app.py	data  rag_demo_project.zip  requirements.txt  src


🥇 الخطوة 3: تهيئة Git

In [26]:
!git init
!git add .
!git commit -m "RAG vs Long Context Demo (FAISS + OpenRouter)"

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/rag_long_context_demo/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@5eaf09e77ea9.(none)')


**🥇 الخطوة 4: أدخل بياناتك**

In [27]:
!git config --global user.name "Your Name"
!git config --global user.email "your@email.com"